# NB03 — Official fixed Q1–Q10 Retrieval Benchmark (Corrected)

**Benchmark:** frozen Verified Reference v1 (2026-10-08). Questions are loaded verbatim from `data/benchmark_questions_v1.json`; do not rewrite them.

**Inputs:** NB02 outputs `data/chunks_naive.json`, `data/chunks_legal_aware.json`. **Outputs:** `data/retrieval_results_nb03_official_v1.csv` and related diagnostic files. No Gemini calls.


In [ ]:
# 필요한 라이브러리 설치
%pip install -q sentence-transformers pandas numpy


In [1]:
from pathlib import Path
import json
import re
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

DATA_DIR = Path("data")
NAIVE_PATH = DATA_DIR / "chunks_naive.json"
LEGAL_PATH = DATA_DIR / "chunks_legal_aware.json"

for path in [NAIVE_PATH, LEGAL_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"{path} 파일을 찾을 수 없습니다. NB02 v2를 Run All한 뒤 JSON 파일을 이 노트북과 같은 폴더에 두세요."
        )

with NAIVE_PATH.open(encoding="utf-8") as f:
    naive_chunks = json.load(f)
with LEGAL_PATH.open(encoding="utf-8") as f:
    legal_chunks = json.load(f)

print(f"Naive chunks: {len(naive_chunks):,}")
print(f"Legal-aware chunks: {len(legal_chunks):,}")
print("Naive sample keys:", list(naive_chunks[0].keys()))
print("Legal-aware sample keys:", list(legal_chunks[0].keys()))

# v2 QA guard: stop if the consolidated Turkish dried-fig entry is not present as one complete logical chunk.
turkey_chunks = [
    c for c in legal_chunks
    if "2019_1793" in c.get("file", "")
    and "Türkiye" in c.get("text", "")
    and "dried figs" in c.get("text", "").lower()
    and "ochratoxin" in c.get("text", "").lower()
    and "30%" in c.get("text", "")
]
print(f"Complete Türkiye dried-fig logical chunks detected: {len(turkey_chunks)}")
if len(turkey_chunks) != 1:
    print("WARNING: Expected exactly one complete Türkiye dried-fig logical chunk. Check that NB02 v2 was rerun and the JSON output was overwritten.")
else:
    print("PASS: Türkiye dried-fig scope, hazards, and 30% check frequency are grouped in one chunk.")



Naive chunks: 342
Legal-aware chunks: 1,383
Naive sample keys: ['chunk_id', 'strategy', 'file', 'regulation_title', 'document_type', 'version_date', 'canonical_url', 'location', 'source_type', 'text', 'char_count']
Legal-aware sample keys: ['chunk_id', 'strategy', 'file', 'regulation_title', 'document_type', 'version_date', 'canonical_url', 'source_type', 'location', 'article_id', 'article', 'heading', 'table_index', 'row_index', 'text', 'char_count']
Complete Türkiye dried-fig logical chunks detected: 1
PASS: Türkiye dried-fig scope, hazards, and 30% check frequency are grouped in one chunk.


## 1. Official benchmark questions — immutable source

The benchmark JSON must be copied into `data/`. The notebook checks its ID, Q1–Q10, and fingerprint. The expected file and term diagnostics below are **weak heuristics**, not a Verified Reference correctness score. Q5–Q10 require multiple legal provisions and may exceed Top-5 retrieval capacity.


In [2]:
import hashlib
BENCHMARK_PATH = DATA_DIR / "benchmark_questions_v1.json"
if not BENCHMARK_PATH.exists():
    raise FileNotFoundError(f"Copy the supplied benchmark_questions_v1.json into {BENCHMARK_PATH}")
benchmark = json.loads(BENCHMARK_PATH.read_text(encoding="utf-8"))
assert benchmark["benchmark_id"] == "regulatory_agent_v1_notion_2026-10-08"
questions = benchmark["questions"]
assert list(questions) == [f"Q{i}" for i in range(1, 11)]
assert all(isinstance(q, str) and q.strip() for q in questions.values())
BENCHMARK_SHA256 = hashlib.sha256(json.dumps(questions, ensure_ascii=False, sort_keys=True).encode("utf-8")).hexdigest()
# Diagnostic hints only; NOT gold-standard labels, not exhaustive requirements.
hints = {
    "Q1": (["EU_2023_915", "EU_2019_1793", "EU_2002_178", "EU_2017_625"], ["aflatoxin", "fig"]),
    "Q2": (["EU_2023_915"], ["6,0", "6.0", "10,0", "10.0"]),
    "Q3": (["EU_2023_915"], ["1.1.3", "sorting", "80"]),
    "Q4": (["EU_2019_1793"], ["fig", "ochratoxin", "30%"]),
    "Q5": (["EU_2019_1793", "EU_2017_625", "EU_2019__1013"], ["certificate", "sampling", "CHED"]),
    "Q6": (["EU_2002_178", "EU_2017_625", "EU_2019_1793"], ["operator", "authority"]),
    "Q7": (["EU_2017_625", "EU_2002_178"], ["66", "withdrawal"]),
    "Q8": (["EU_2023_915", "EU_2019_1793"], ["annex", "amendment"]),
    "Q9": (["EU_2023_915", "EU_2019_1793", "EU_2017_625", "EU_2002_178"], ["article", "annex"]),
    "Q10": (["EU_2023_915", "EU_2019_1793", "EU_2017_625", "EU_2002_178"], ["fig", "aflatoxin"]),
}
queries = [{"qid": qid, "question": question, "expected_files": hints[qid][0],
            "expected_terms": hints[qid][1], "why": "Exploratory diagnostic only"}
           for qid, question in questions.items()]
print("BENCHMARK", benchmark["benchmark_id"], "SHA256", BENCHMARK_SHA256)
display(pd.DataFrame([{"qid": q["qid"], "question": q["question"]} for q in queries]))


BENCHMARK regulatory_agent_v1_notion_2026-10-08 SHA256 2ee3bf32133c31bc62514cfa85e116bd5dd846a89b282a1a6f62f1ec849c3e4e


,qid,question
0,Q1,A shipment of dried figs originating from Türk...
1,Q2,What is the applicable EU maximum level for af...
2,Q3,Does the applicable limit change depending on ...
3,Q4,Are dried figs from Türkiye subject to increas...
4,Q5,"If so, what controls, sampling/testing require..."
5,Q6,Which authorities or food business operators a...
6,Q7,What regulatory actions may follow if aflatoxi...
7,Q8,Which parts of the assessment depend on curren...
8,Q9,Provide the legal basis for each major conclus...
9,Q10,"Based on the evidence above, prepare a concise..."


## 2. 모델 로드 및 임베딩 생성

두 청킹 방식 모두 **동일한 모델**을 사용한다. 이 비교에서 달라지는 것은 청킹 전략뿐이다. 임베딩은 한 번 계산하고, 이후 코사인 유사도로 Top-k를 구한다.


In [3]:
MODEL_NAME = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
model = SentenceTransformer(MODEL_NAME)
print("Loaded embedding model:", MODEL_NAME)

def chunk_text(chunk):
    # Legal-aware chunks include location metadata; add it to the text the retriever sees.
    metadata_parts = [
        chunk.get("regulation_title", ""),
        chunk.get("location", ""),
        chunk.get("article", ""),
        chunk.get("heading", ""),
    ]
    metadata = " | ".join(str(x) for x in metadata_parts if x)
    return (metadata + "\n" + chunk.get("text", "")).strip()

naive_texts = [chunk_text(c) for c in naive_chunks]
legal_texts = [chunk_text(c) for c in legal_chunks]
query_texts = [q["question"] for q in queries]

naive_embeddings = model.encode(
    naive_texts, convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=True
)
legal_embeddings = model.encode(
    legal_texts, convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=True
)
query_embeddings = model.encode(
    query_texts, convert_to_numpy=True, normalize_embeddings=True,
    show_progress_bar=False
)

print("Embedding shapes:")
print("  queries:", query_embeddings.shape)
print("  naive:  ", naive_embeddings.shape)
print("  legal:  ", legal_embeddings.shape)


'(ProtocolError('Connection aborted.', ConnectionResetError(10054, '현재 연결은 원격 호스트에 의해 강제로 끊겼습니다', None, 10054, None)), '(Request ID: 7f5407fe-d894-4dca-aff4-203021728186)')' thrown while requesting HEAD https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, '현재 연결은 원격 호스트에 의해 강제로 끊겼습니다', None, 10054, None)), '(Request ID: b3cc90bd-34b3-4c12-976d-f02cab241216)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2/e8f8c211226b894fcb81acc59f3b34ba3efd5f42/modules.json
Retrying in 1s [Retry 1/5].
'(ProtocolError('Connection aborted.', ConnectionResetError(10054, '현재 연결은 원격 호스트에 의해 강제로 끊겼습니다', None, 10054, None)), '(Request ID: b4292498-1796-48a5-b8ba-de47f678096a)')' thrown while requesting HEAD https://huggingface.co/api/resolve-cache/models/sentence-trans

Loaded embedding model: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Batches:   0%|          | 0/44 [00:00<?, ?it/s]

Embedding shapes:
  queries: (10, 384)
  naive:   (342, 384)
  legal:   (1383, 384)


## 3. Top-5 검색 결과

검색 점수는 코사인 유사도다. 점수는 관련성의 확률이 아니며, 서로 다른 질의 사이의 절대 점수 비교보다 같은 질의에서 나온 순위와 실제 근거의 내용을 보는 것이 중요하다.


In [4]:
TOP_K = 5

def retrieve(chunks, texts, embeddings, q, q_embedding, strategy, top_k=TOP_K):
    scores = embeddings @ q_embedding
    top_indices = np.argsort(scores)[::-1][:top_k]
    rows = []
    for rank, idx in enumerate(top_indices, start=1):
        c = chunks[int(idx)]
        rows.append({
            "qid": q["qid"],
            "question": q["question"],
            "benchmark_id": benchmark["benchmark_id"],
            "benchmark_sha256": BENCHMARK_SHA256,
            "strategy": strategy,
            "rank": rank,
            "score": float(scores[idx]),
            "chunk_id": c.get("chunk_id", ""),
            "file": c.get("file", ""),
            "version_date": c.get("version_date", ""),
            "location": c.get("location", ""),
            "article": c.get("article", ""),
            "heading": c.get("heading", ""),
            "text_preview": re.sub(r"\s+", " ", c.get("text", ""))[:650],
        })
    return rows

all_results = []
for i, q in enumerate(queries):
    all_results.extend(retrieve(
        naive_chunks, naive_texts, naive_embeddings, q, query_embeddings[i], "naive"
    ))
    all_results.extend(retrieve(
        legal_chunks, legal_texts, legal_embeddings, q, query_embeddings[i], "legal_aware"
    ))

results_df = pd.DataFrame(all_results)
display(results_df[results_df["rank"] <= 3][
    ["qid", "strategy", "rank", "score", "file", "location", "text_preview"]
].sort_values(["qid", "strategy", "rank"]).reset_index(drop=True))


,qid,strategy,rank,score,file,location,text_preview
0,Q1,legal_aware,1,0.631073,EU_2019_1793_consolidated_2026-06-30.html,"Annex II, Section 1, Row 14 — Türkiye dried-fi...",COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
1,Q1,legal_aware,2,0.519528,EU_2023_915_consolidated_2026-08-19.html,"Table 4, row 3",COMMISSION REGULATION (EU) 2023/915. Table con...
2,Q1,legal_aware,3,0.519346,EU_2023_915_consolidated_2026-08-19.html,"Table 3, row 3",COMMISSION REGULATION (EU) 2023/915. Table con...
3,Q1,naive,1,0.593636,EU_2019_1793_consolidated_2026-06-30.html,fixed_chunk_15,7 99 39 | 05; 06 ex 2007 99 50 | 33 ex 2007 99...
4,Q1,naive,2,0.572308,EU_2019_1793_consolidated_2026-06-30.html,fixed_chunk_20,40 ex 2008 19 19 | 50 ex 2008 19 92 | 40 | Afl...
5,Q1,naive,3,0.549897,EU_2019_1793_consolidated_2026-06-30.html,fixed_chunk_5,the analyses shall be performed in accordance ...
6,Q10,legal_aware,1,0.740259,EU_2017_625_consolidated_2025-01-05.html,Article 148 — Relation with Regulations (EC) N...,Article 148 Relation with Regulations (EC) No ...
7,Q10,legal_aware,2,0.701411,EU_2019_1793_consolidated_2026-06-30.html,Article 3 — Sampling and analyses,Article 3 Sampling and analyses The sampling a...
8,Q10,legal_aware,3,0.693205,EU_2017_625_consolidated_2025-01-05.html,Article 98 — Responsibilities and tasks of Eur...,"ce materials, to be used to detect violations ..."
9,Q10,naive,1,0.757784,EU_2017_625_consolidated_2025-01-05.html,fixed_chunk_170,tion of a European Union reference laboratory ...


## 4. 간단한 자동 점검 지표

아래 지표는 **초기 진단용 휴리스틱**이다.

- **Expected-file Hit@5:** Top-5 중 질문에 지정된 법령 파일이 하나라도 있는가?
- **Expected-term coverage@5:** Top-5 전체 텍스트에 초기 기준 용어 중 하나 이상이 있는가?
- **Both checks:** 두 조건을 모두 만족하는가?

이 기준은 완전한 relevance judgment가 아니다. 예를 들어 올바른 법령이 검색되어도 조문 번호가 해당 chunk에 없을 수 있고, 같은 단어가 나와도 질문에 답하는 근거가 아닐 수 있다. 따라서 이 점수만으로 청킹 전략의 승자를 선언하지 않는다.


In [5]:
def normalize_text(value):
    return re.sub(r"\s+", " ", str(value or "")).casefold()

metric_rows = []
for q in queries:
    for strategy in ["naive", "legal_aware"]:
        top = results_df[
            (results_df["qid"] == q["qid"]) &
            (results_df["strategy"] == strategy) &
            (results_df["rank"] <= TOP_K)
        ]
        files_text = " ".join(top["file"].fillna("").astype(str).tolist())
        body_text = " ".join(top["text_preview"].fillna("").astype(str).tolist())
        file_hit = any(expected.casefold() in files_text.casefold() for expected in q["expected_files"])
        term_hit = any(term.casefold() in normalize_text(body_text) for term in q["expected_terms"])
        metric_rows.append({
            "qid": q["qid"],
            "strategy": strategy,
            "expected_file_hit_at_5": file_hit,
            "expected_term_coverage_at_5": term_hit,
            "both_checks": file_hit and term_hit,
        })

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df)

summary_df = metrics_df.groupby("strategy", as_index=False).agg(
    expected_file_hit_at_5=("expected_file_hit_at_5", "sum"),
    expected_term_coverage_at_5=("expected_term_coverage_at_5", "sum"),
    both_checks=("both_checks", "sum"),
)
summary_df["total_queries"] = len(queries)
for col in ["expected_file_hit_at_5", "expected_term_coverage_at_5", "both_checks"]:
    summary_df[col + "_rate"] = (summary_df[col] / len(queries)).round(3)
display(summary_df)


,qid,strategy,expected_file_hit_at_5,expected_term_coverage_at_5,both_checks
0,Q1,naive,True,True,True
1,Q1,legal_aware,True,True,True
2,Q2,naive,True,True,True
3,Q2,legal_aware,True,True,True
4,Q3,naive,True,False,False
5,Q3,legal_aware,True,True,True
6,Q4,naive,True,True,True
7,Q4,legal_aware,True,True,True
8,Q5,naive,True,False,False
9,Q5,legal_aware,True,True,True


,strategy,expected_file_hit_at_5,expected_term_coverage_at_5,both_checks,total_queries,expected_file_hit_at_5_rate,expected_term_coverage_at_5_rate,both_checks_rate
0,legal_aware,9,8,8,10,0.9,0.8,0.8
1,naive,8,7,6,10,0.8,0.7,0.6


## 5. 핵심 케이스 점검: Türkiye dried figs

Q4와 Q9는 특히 중요하다. `2019/1793`에서 제품 범위, aflatoxins, ochratoxin A, 30% 검사 빈도가 **한 논리적 항목 안에 함께 검색되는지** 직접 확인한다.


In [6]:
def show_query_results(qid, strategy, top_k=TOP_K):
    cols = ["rank", "score", "file", "version_date", "location", "article", "text_preview"]
    subset = results_df[
        (results_df["qid"] == qid) &
        (results_df["strategy"] == strategy) &
        (results_df["rank"] <= top_k)
    ][cols]
    print(f"\n{qid} — {strategy}")
    display(subset.reset_index(drop=True))

show_query_results("Q4", "naive")
show_query_results("Q4", "legal_aware")
show_query_results("Q9", "legal_aware")



Q4 — naive


,rank,score,file,version_date,location,article,text_preview
0,1,0.560604,EU_2017_625_consolidated_2025-01-05.html,2025-01-05,fixed_chunk_70,,"rticle 1(2), the competent authorities shall p..."
1,2,0.545745,EU_2017_625_consolidated_2025-01-05.html,2025-01-05,fixed_chunk_105,,"riginating from, and returning to, the Union f..."
2,3,0.519422,EU_2017_625_consolidated_2025-01-05.html,2025-01-05,fixed_chunk_68,,"ithstanding paragraphs 1 and 3, the competent ..."
3,4,0.510300,EU_2019_1793_consolidated_2026-06-30.html,2026-06-30,fixed_chunk_23,,| Salmonella ( 5 ) | 50 (Food) | ex 2008 19 19...
4,5,0.509919,EU_2017_625_consolidated_2025-01-05.html,2025-01-05,fixed_chunk_48,,tivities performed by the competent authoritie...



Q4 — legal_aware


,rank,score,file,version_date,location,article,text_preview
0,1,0.683301,EU_2019_1793_consolidated_2026-06-30.html,2026-06-30,"Annex II, Section 1, Row 14 — Türkiye dried-fi...",None,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
1,2,0.534552,EU_2017_625_consolidated_2025-01-05.html,2025-01-05,Article 73 — Approval of pre-export controls p...,Article 73,may only be granted to a third country if the ...
2,3,0.524941,EU_2019_1793_consolidated_2026-06-30.html,2026-06-30,"Table 5, row 169",None,COMMISSION IMPLEMENTING REGULATION (EU) 2019/1...
3,4,0.518348,EU_2017_625_consolidated_2025-01-05.html,2025-01-05,"Article 74 — Non-compliance with, and withdraw...",Article 74,"Article 74 Non-compliance with, and withdrawal..."
4,5,0.516341,EU_2023_915_consolidated_2026-08-19.html,2026-08-19,"Table 4, row 3",None,COMMISSION REGULATION (EU) 2023/915. Table con...



Q9 — legal_aware


,rank,score,file,version_date,location,article,text_preview
0,1,0.552966,EU_2017_625_consolidated_2025-01-05.html,2025-01-05,Article 140 — Reporting of infringements,Article 140,Article 140 Reporting of infringements 1. Memb...
1,2,0.551398,EU_2017_625_consolidated_2025-01-05.html,2025-01-05,Article 1 — Subject matter and scope,Article 1,ith Article 118(1) of that Regulation. ▼B 5. A...
2,3,0.550503,EU_2017_625_consolidated_2025-01-05.html,2025-01-05,Article 55 — Decisions on consignments,Article 55,Article 55 Decisions on consignments 1. A deci...
3,4,0.550016,EU_2019_1793_consolidated_2026-06-30.html,2026-06-30,Article 3 — Sampling and analyses,Article 3,Article 3 Sampling and analyses The sampling a...
4,5,0.549114,EU_2002_178_consolidated_2026-01-01.html,2026-01-01,Article 29 — Scientific opinions,Article 29,e request. ▼M8 6. In order to apply this Artic...


## 6. Save corrected official benchmark retrieval results

**Do not overwrite** the previous exploratory NB03 files. Every row records the original question and benchmark SHA256. This file is the only valid input to corrected NB04.


In [7]:
results_path = DATA_DIR / "retrieval_results_nb03_official_v1.csv"
metrics_path = DATA_DIR / "retrieval_metrics_nb03_official_v1.csv"
summary_path = DATA_DIR / "retrieval_summary_nb03_official_v1.csv"

results_df.to_csv(results_path, index=False, encoding="utf-8-sig")
metrics_df.to_csv(metrics_path, index=False, encoding="utf-8-sig")
summary_df.to_csv(summary_path, index=False, encoding="utf-8-sig")

print("Saved:")
print(" -", results_path)
print(" -", metrics_path)
print(" -", summary_path)


Saved:
 - data\retrieval_results_nb03_official_v1.csv
 - data\retrieval_metrics_nb03_official_v1.csv
 - data\retrieval_summary_nb03_official_v1.csv


In [ ]:
assert set(results_df["qid"]) == set(questions)
assert all(results_df.groupby(["qid", "strategy"]).size().eq(TOP_K))
assert all(results_df.apply(lambda r: r["question"] == questions[r["qid"]], axis=1))
assert results_df["benchmark_sha256"].nunique() == 1
print("PASS: original Notion Q1–Q10, matching retrieval questions, 5 hits per strategy")


## 7. 사람이 확인할 체크리스트

아래 항목을 Q1–Q10 각각에 대해 검토한 후, 별도 회고/표에 기록한다.

- [ ] Top-5에 질문에 답하는 실제 법령 근거가 있는가?
- [ ] 조문/부속서/표의 위치가 식별되는가?
- [ ] 한 청크에 답변에 필요한 핵심 조건과 예외가 함께 들어 있는가?
- [ ] 법령 제목, 버전 날짜, 출처 URL 등 추적 정보가 유지되는가?
- [ ] 같은 질의에서 naive와 legal-aware 중 어느 쪽이 더 적합한 근거를 상위에 가져왔는가?
- [ ] 근거가 누락되었을 때 청킹 문제인지, 임베딩/질의 문제인지, 원천 파싱 문제인지 구분했는가?

### NB03 완료 기준
1. 두 청킹 방식 모두 Q1–Q10의 Top-5를 생성했다.
2. 자동 지표와 Top-5 원문을 함께 검토했다.
3. 핵심 Q4/Q9에서 Türkiye dried-fig 항목의 범위·위험요소·30% 빈도가 한 청크에 보존되고 검색되는지 확인했다.
4. 검색 실패 사례를 기록한 뒤에만 NB04(LLM 기반 답변 생성)로 넘어간다.

**해석 원칙:** legal-aware가 항상 이긴다고 미리 가정하지 않는다. 검색 결과의 근거 정확성, 문맥 완결성, 출처 추적 가능성을 보고 판단한다.
